# Solution 05 — deps carrying an HTTP client

`WeatherDeps` holds the `httpx.AsyncClient`; tools reach it via `ctx.deps.client`.
Both Open-Meteo endpoints are free and keyless — the only secret here is none.
(Needs network access; that's the point of a *real* client in deps.)

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

In [ ]:
from dataclasses import dataclass

import httpx
from pydantic_ai import Agent, ModelRetry, RunContext


@dataclass
class WeatherDeps:
    client: httpx.AsyncClient


weather_agent = Agent(
    MODEL,
    deps_type=WeatherDeps,
    instructions="You compare current weather between cities. Use your tools for ALL lookups.",
)


@weather_agent.tool
async def geocode(ctx: RunContext[WeatherDeps], city: str) -> dict:
    """Find the latitude/longitude of a city."""
    url = "https://geocoding-api.open-meteo.com/v1/search"
    response = await ctx.deps.client.get(url, params={"name": city, "count": 1})
    if response.status_code != 200:
        raise ModelRetry(f"Geocoding failed ({response.status_code}) — try again or another city.")
    results = response.json().get("results") or []
    if not results:
        raise ModelRetry(
            f"Could not geocode {city!r} — ask the user for the country or try a bigger city."
        )
    hit = results[0]
    return {"name": hit["name"], "latitude": hit["latitude"], "longitude": hit["longitude"]}


@weather_agent.tool
async def current_temp(ctx: RunContext[WeatherDeps], latitude: float, longitude: float) -> float:
    """Get the current temperature (°C) at a location."""
    url = "https://api.open-meteo.com/v1/forecast"
    response = await ctx.deps.client.get(
        url, params={"latitude": latitude, "longitude": longitude, "current": "temperature_2m"}
    )
    if response.status_code != 200:
        raise ModelRetry(f"Weather lookup failed ({response.status_code}) — retry.")
    return response.json()["current"]["temperature_2m"]


async with httpx.AsyncClient() as client:
    result = await weather_agent.run(
        "Is it warmer right now: Paris or Tokyo? Give both temperatures.",
        deps=WeatherDeps(client),
    )
print(result.output)

**Notes**

- The client is created *outside* and injected — so a test can inject a fake one
  (notebook 08 thinking) and never touch the network.
- `ModelRetry` covers both transport failures *and* logical failures (city not found):
  the model can recover from either.
- `async with httpx.AsyncClient()` — the deps pattern composes with normal async
  resource management. This exact shape returns in the FastAPI phase, per request.